# Consent and Transparency Standards for Learning Analytics in Philippine HEIs

### Reproducible analysis — Group 1, Theme 2

**Authors:** Christan Ray M. Aningat, Chryz Johnred C. Dumalagan, and Axel C. Magallanes  
**Course:** Social and Professional Issues, Technological University of the Philippines  
**Paper:** *Consent and Transparency Standards for Learning Analytics in Philippine HEIs* (S4)

This notebook downloads the public Open University Learning Analytics Dataset (OULAD), runs the tested analysis script `G1_Theme2_S4_Analysis.py`, and reproduces Tables I–IV of the paper.

**To run:** *Kernel → Restart & Run All*. The first run downloads about 45 MB and takes a few minutes.

## What this notebook reproduces

- **Table I** — auditable Day-28 cleaning log and cohort counts.
- **Table II** — descriptive statistics for the cohort and key subgroups.
- **Table III** — chi-square tests, odds ratios, and Cramér's V.
- **Table IV** — full, minimized, and demographic-only model audit (AUC, accuracy, EOD, AOD).

The script downloads OULAD from the UCI Machine Learning Repository (ID 349, CC BY 4.0). It does not use restricted or personally identifying data.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
cmd = [
    sys.executable,
    "G1_Theme2_S4_Analysis.py",
    "--download",
    "--data-dir", "data/oulad",
    "--out", "results.json",
]
print("Running:", " ".join(cmd))
completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
print(completed.stdout.strip().splitlines()[-1])

Running: C:\Users\xel\AppData\Local\Temp\opencode\spi-audit-venv\Scripts\python.exe G1_Theme2_S4_Analysis.py --download --data-dir data/oulad --out results.json


Saved results.json


In [3]:
results = json.loads(Path("results.json").read_text(encoding="utf-8"))
results["cleaning_log"]

{'raw': 32593,
 'missing_registration_date': 45,
 'late_registration': 18,
 'pre_cutoff_withdrawal': 5002,
 'cohort': 27528,
 'unique_learners': 24835,
 'at_risk_n': 12150,
 'at_risk_pct': 44.14,
 'vle_rows_cohort': 2763370,
 'clicks_cohort': 9827686,
 'assessment_rows_cohort': 25081,
 'grade_n': 20398,
 'imd_missing': 1028,
 'imd_missing_pct': 3.73,
 'known_imd': 26500,
 'activity_types': 18}

### Table I — Day-28 cleaning log

In [4]:
log = results["cleaning_log"]
steps = [
    ("1. Raw registrations", "Load all studentInfo registrations", log["raw"]),
    ("2. Missing registration date", "Exclude records with no date_registration", -log["missing_registration_date"]),
    ("3. Late registration", "Exclude date_registration >= 28", -log["late_registration"]),
    ("4. Pre-cutoff withdrawal", "Exclude date_unregistration < 28", -log["pre_cutoff_withdrawal"]),
]
rows = []
running = 0
for step, rule, delta in steps:
    running = delta if step.startswith("1.") else running + delta
    rows.append({"Step": step, "Rule": rule, "Count": delta, "Cohort n": running})
rows.append({"Step": "5. Final cohort", "Rule": "Active Day-28 registrations", "Count": log["cohort"], "Cohort n": log["cohort"]})
pd.DataFrame(rows)

,Step,Rule,Count,Cohort n
0,1. Raw registrations,Load all studentInfo registrations,32593,32593
1,2. Missing registration date,Exclude records with no date_registration,-45,32548
2,3. Late registration,Exclude date_registration >= 28,-18,32530
3,4. Pre-cutoff withdrawal,Exclude date_unregistration < 28,-5002,27528
4,5. Final cohort,Active Day-28 registrations,27528,27528


### Table II — Descriptive statistics

In [5]:
labels = [
    ("all", "All active learners"),
    ("imd_0_10", "IMD 0–10%"),
    ("imd_40_50", "IMD 40–50%"),
    ("imd_90_100", "IMD 90–100%"),
    ("disability_y", "Disability Y"),
    ("disability_n", "Disability N"),
    ("age_0_35", "Age 0–35"),
    ("age_35_plus", "Age 35–55/55+"),
    ("male", "Male"),
    ("female", "Female"),
]
rows = []
for key, label in labels:
    t = results["tables"][key]
    rows.append({
        "Subgroup": label,
        "n": t["n"],
        "Share %": t["share_pct"],
        "Clicks mean/SD": f'{t["clicks_mean"]:.1f}/{t["clicks_sd"]:.1f}',
        "Clicks median (IQR)": f'{t["clicks_median"]:.1f} ({t["clicks_iqr"]:.1f})',
        "Grade mean/SD": f'{t["grade_mean"]:.1f}/{t["grade_sd"]:.1f}',
        "At risk n (%)": f'{t["risk_n"]} ({t["risk_pct"]:.2f})',
    })
pd.DataFrame(rows)

,Subgroup,n,Share %,Clicks mean/SD,Clicks median (IQR),Grade mean/SD,At risk n (%)
0,All active learners,27528,100.00,357.0/430.9,225.0 (380.0),73.1/21.6,12150 (44.14)
1,IMD 0–10%,2647,9.62,309.8/415.1,178.0 (342.0),70.6/22.2,1484 (56.06)
2,IMD 40–50%,2713,9.86,337.3/410.0,202.0 (358.0),72.7/21.7,1196 (44.08)
3,IMD 90–100%,2240,8.14,393.8/479.2,264.0 (376.2),76.3/19.5,781 (34.87)
4,Disability Y,2610,9.48,334.5/438.5,198.5 (355.8),71.5/21.6,1404 (53.79)
5,Disability N,24918,90.52,359.4/430.0,228.0 (382.0),73.3/21.6,10746 (43.13)
6,Age 0–35,19172,69.65,320.2/369.6,209.0 (340.2),72.8/21.2,8843 (46.12)
7,Age 35–55/55+,8356,30.35,441.6/536.6,275.0 (477.2),73.7/22.3,3307 (39.58)
8,Male,15169,55.10,418.1/457.5,290.0 (419.0),75.0/18.6,6915 (45.59)
9,Female,12359,44.90,282.0/382.7,158.0 (294.0),70.7/24.5,5235 (42.36)


### Table III — Chi-square tests

In [6]:
rows = []
for key, label in [("imd", "IMD band"), ("disability", "Disability"), ("gender", "Gender")]:
    t = results["tests"][key]
    rows.append({
        "Test": label,
        "n": t["n"],
        "chi2": t["chi2"],
        "df": t["df"],
        "p": f'{t["p"]:.3e}',
        "Cramer's V": t["cramers_v"],
        "Odds ratio": t["odds_ratio"],
        "95% CI": f'{t["ci_low"]}–{t["ci_high"]}',
        "Rate A %": t["rate_a_pct"],
        "Rate B %": t["rate_b_pct"],
    })
pd.DataFrame(rows)

,Test,n,chi2,df,p,Cramer's V,Odds ratio,95% CI,Rate A %,Rate B %
0,IMD band,26500,413.33,9,1.961e-83,0.1249,2.384,2.123–2.677,56.06,34.87
1,Disability,27528,108.61,1,1.977e-25,0.0628,1.535,1.416–1.665,53.79,43.13
2,Gender,27528,28.66,1,8.623e-08,0.0323,1.140,1.087–1.196,45.59,42.36


### Table IV — Model audit

In [7]:
rows = []
for m in results["models"]:
    rows.append({
        "Model": m["model"],
        "Inputs": m["inputs"],
        "AUC": m["auc"],
        "Accuracy %": m["accuracy_pct"],
        "EOD IMD": m["eod_imd"],
        "AOD IMD": m["aod_imd"],
        "EOD Disability": m["eod_disability"],
        "AOD Disability": m["aod_disability"],
    })
pd.DataFrame(rows)

,Model,Inputs,AUC,Accuracy %,EOD IMD,AOD IMD,EOD Disability,AOD Disability
0,Full,29,0.777,71.4,0.219,0.221,0.099,0.102
1,Minimized,3,0.725,67.5,0.092,0.086,0.005,0.027
2,Demographic,4,0.588,57.9,0.879,0.870,0.422,0.411


### Verification against the published results

In [8]:
expected = json.loads(Path("expected_results.json").read_text(encoding="utf-8"))

def assert_close(actual, expected, path=""):
    """Compare every reported value, allowing tiny float differences across platforms."""
    if isinstance(actual, dict):
        assert set(actual) == set(expected), f"Key mismatch at {path}"
        for key in actual:
            assert_close(actual[key], expected[key], f"{path}/{key}")
    elif isinstance(actual, list):
        assert len(actual) == len(expected), f"Length mismatch at {path}"
        for i, (a, b) in enumerate(zip(actual, expected)):
            assert_close(a, b, f"{path}/{i}")
    elif isinstance(actual, float):
        assert abs(actual - expected) <= 1e-9 * max(1.0, abs(expected)), f"Float mismatch at {path}: {actual} vs {expected}"
    else:
        assert actual == expected, f"Mismatch at {path}: {actual} vs {expected}"

assert_close(results, expected)
print("Verified: all reported values match expected_results.json within tolerance.")


Verified: all reported values match expected_results.json within tolerance.


## Notes

- OULAD is a United Kingdom distance-learning dataset. The paper uses it as a design stress test, not as a sample of Filipino students.
- The analysis is deterministic: `HistGradientBoostingClassifier(random_state=42)` and grouped 5-fold cross-validation. Re-running the notebook should reproduce the same JSON.
- Data license: OULAD, UCI Machine Learning Repository ID 349, CC BY 4.0.
- No re-identification was attempted; only aggregate results are reported.